# 01 · Sample construction and descriptive statistics
Builds the analysed sample of Section 2.2: defect-inducing commits (`contains_bug == 1`), with each project's right-censored tail removed. Reproduces the counts in Section 2.2 and the per-project descriptive statistics (Appendix A).

In [1]:
# On Google Colab, uncomment the next line first:
# !pip install -q dowhy==0.14 statsmodels networkx
import warnings, numpy as np, pandas as pd, networkx as nx, statsmodels.api as sm
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

# Data source: Song & Minku (TSE 2022) replication package. Set BASE to a local folder to work offline.
BASE = "https://raw.githubusercontent.com/sunnysong14/ContinualPerformanceValidityTSE2022/main/data/"
Y, T = "days_to_first_fix", "author_date_unix_timestamp"

PROJECTS = {"brackets": "Brackets", "broadleaf": "Broadleaf", "camel": "Camel", "corefx": "CoreFX",
            "django": "Django", "fabric8": "Fabric8", "jgroup": "JGroup", "nova": "Nova", "rails": "Rails",
            "rust": "Rust", "tensorflow": "TensorFlow", "vscode": "VSCode", "wp-calypso": "WP-Calypso"}
# Projects on which the 33-edge graph is informative and not rejected (Section 4.1)
COMPATIBLE = ["brackets", "camel", "corefx", "django", "fabric8", "nova",
              "rails", "rust", "tensorflow", "vscode", "wp-calypso"]
METRICS = [T, "exp", "rexp", "fix", "ns", "nf", "sexp", "lt", "nuc", "ndev", "age", "la", "ld", "entropy", Y]


def load(name, horizon_q=0.9, return_info=False):
    """Defect-inducing commits of one project, with the right-censored tail removed.

    A commit made within `horizon` of the last commit in the data has had less time to be fixed than
    the project's 90th-percentile defect lifetime, so it is excluded (Section 2.2).
    """
    raw = pd.read_csv(f"{BASE}{name}.csv")
    raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")].rename(columns={"entrophy": "entropy"})
    bug = raw[raw.contains_bug == 1]
    horizon_days = bug[Y].quantile(horizon_q)
    kept = bug[bug[T] <= bug[T].max() - horizon_days * 86400]
    df = kept.replace([np.inf, -np.inf], np.nan).dropna(subset=METRICS).reset_index(drop=True)
    df["fix"] = df["fix"].astype(float)
    if return_info:
        return df, dict(project=PROJECTS[name], all_commits=len(raw), defect_inducing=len(bug),
                        horizon_days=round(horizon_days), after_censoring=len(kept), analysed=len(df))
    return df


## Sample size at each step

In [2]:
rows, data = [], {}
for p in PROJECTS:
    data[p], info = load(p, return_info=True)
    rows.append(info)
S = pd.DataFrame(rows)
S["removed_by_censoring_%"] = (100 * (1 - S.after_censoring / S.defect_inducing)).round(1)
S["graph_compatible"] = S.index.map(lambda i: list(PROJECTS)[i] in COMPATIBLE)
S

,project,all_commits,defect_inducing,horizon_days,after_censoring,analysed,removed_by_censoring_%,graph_compatible
0,Brackets,11601,3947,644,3860,3860,2.2,True
1,Broadleaf,12336,2502,890,1990,1990,20.5,False
2,Camel,30229,6248,739,5471,5471,12.4,True
3,CoreFX,26627,1841,565,1286,1286,30.1,True
4,Django,26360,11240,1010,10510,10510,6.5,True
5,Fabric8,12495,2580,383,2561,2561,0.7,True
6,JGroup,18003,3147,1277,3100,3100,1.5,False
7,Nova,26313,11667,568,11011,11011,5.6,True
8,Rails,57949,14860,1199,13088,13086,11.9,True
9,Rust,73876,1492,465,1422,1422,4.7,True


In [3]:
c = S[S.graph_compatible]
print(f"11 compatible projects: {c.defect_inducing.sum():,} defect-inducing commits, "
      f"{c.after_censoring.sum():,} after censoring, {c.analysed.sum():,} analysed")
print(f"Horizon: {c.horizon_days.min()}–{c.horizon_days.max()} days; removed "
      f"{c['removed_by_censoring_%'].min()}–{c['removed_by_censoring_%'].max()}% per project, "
      f"{100 * (1 - c.after_censoring.sum() / c.defect_inducing.sum()):.1f}% overall")

11 compatible projects: 78,317 defect-inducing commits, 68,754 after censoring, 68,752 analysed
Horizon: 280–1199 days; removed 0.7–30.1% per project, 12.2% overall


## Descriptive statistics of the model variables (analysed sample)

In [4]:
def describe(df, cols=("ndev", "entropy", "sexp", "rexp", "exp", "ns", "nf", Y)):
    out = []
    for v in cols:
        x = df[v]; se = x.std() / np.sqrt(len(x))
        out.append(dict(variable=v, mean=x.mean(), median=x.median(), sd=x.std(),
                        ci_low=x.mean() - 1.96 * se, ci_high=x.mean() + 1.96 * se))
    return pd.DataFrame(out).round(3)

for p in COMPATIBLE:
    print(f"\n{PROJECTS[p]} (n = {len(data[p]):,})")
    display(describe(data[p]))


Brackets (n = 3,860)


,variable,mean,median,sd,ci_low,ci_high
0,ndev,37.844,34.000,23.952,37.088,38.599
1,entropy,0.759,0.578,0.885,0.731,0.787
2,sexp,371.257,173.000,496.883,355.582,386.933
3,rexp,19.271,1.601,87.380,16.515,22.028
4,exp,751.916,333.750,1060.564,718.458,785.374
5,ns,1.207,1.000,0.422,1.194,1.220
6,nf,3.516,2.000,11.283,3.160,3.872
7,days_to_first_fix,174.699,14.949,343.558,163.860,185.537



Camel (n = 5,471)


,variable,mean,median,sd,ci_low,ci_high
0,ndev,24.789,16.000,22.832,24.184,25.394
1,entropy,1.840,1.673,1.387,1.804,1.877
2,sexp,4645.476,1709.000,6523.724,4472.606,4818.345
3,rexp,31.182,1.023,415.335,20.176,42.187
4,exp,12542.952,6444.000,15276.617,12138.143,12947.761
5,ns,1.347,1.000,0.736,1.327,1.366
6,nf,10.120,5.000,31.998,9.272,10.968
7,days_to_first_fix,262.891,42.707,472.048,250.382,275.400



CoreFX (n = 1,286)


...

,variable,mean,median,sd,ci_low,ci_high
0,ndev,201.634,199.000,111.429,199.713,203.555
1,entropy,1.514,1.308,1.247,1.493,1.536
2,sexp,19009.295,817.000,30372.726,18485.686,19532.905
3,rexp,7.940,1.095,195.883,4.563,11.317
4,exp,19384.211,944.500,30626.902,18856.220,19912.203
5,ns,1.098,1.000,0.418,1.091,1.105
6,nf,9.835,4.000,51.947,8.939,10.731
7,days_to_first_fix,114.656,46.971,166.513,111.786,117.527



VSCode (n = 1,023)


,variable,mean,median,sd,ci_low,ci_high
0,ndev,42.853,43.000,22.668,41.464,44.242
1,entropy,1.231,1.033,1.148,1.161,1.302
2,sexp,2415.135,552.000,3872.347,2177.838,2652.432
3,rexp,24.503,1.233,150.329,15.291,33.715
4,exp,4756.988,3495.000,4927.453,4455.034,5058.942
5,ns,1.349,1.000,0.681,1.307,1.391
6,nf,7.450,3.000,41.915,4.881,10.018
7,days_to_first_fix,128.128,41.066,191.864,116.370,139.885



WP-Calypso (n = 5,596)


,variable,mean,median,sd,ci_low,ci_high
0,ndev,108.299,113.000,57.412,106.794,109.803
1,entropy,1.319,1.148,1.154,1.288,1.349
2,sexp,421.451,201.000,651.751,404.374,438.527
3,rexp,27.594,1.072,241.864,21.257,33.931
4,exp,553.350,291.250,832.246,531.544,575.155
5,ns,1.184,1.000,0.489,1.171,1.197
6,nf,6.700,3.000,45.459,5.509,7.891
7,days_to_first_fix,226.315,79.090,293.813,218.616,234.013
